# ser_client-api + ser_client-ftps - Demo against ser_server-ftps

Demonstrates the client-side stack against the mock FTPS server:

1. **ser_client-api**: parse a GLeaves prescription JSON into domain model objects (`CompositionData`), generate an HL7v2 ORU_R01 message, and parse the ACK response
2. **ser_client-ftps**: transfer files to `ser_server-ftps` and retrieve the ACK file

**Prerequisites**

- `ser_server-ftps` repository checked out
- Client certificates: `client-cert.pem`, `client-key.pem`, `ca-cert.pem`
- Both packages installed:
  ```bash
  pip install git+https://github.com/GenomeCAD/ser_client-api.git@v0.5.0
  pip install git+https://github.com/GenomeCAD/ser_client-ftps.git@v0.3.0
  ```

**Start the mock server**

In the `ser_server-ftps` root directory:
```bash
docker compose up -d
```

Mock server: `localhost:990`, credentials: `CHU-TEST` / `ftppassword`

### Optional: install required packages

Restart the kernel after the installation is complete.

In [ ]:
import sys
!{sys.executable} -m pip install --force-reinstall git+https://github.com/GenomeCAD/ser_client-api.git@v0.5.0
!{sys.executable} -m pip install --force-reinstall git+https://github.com/GenomeCAD/ser_client-ftps.git@v0.3.0

## 0. Configuration

In [ ]:
FTPS_HOST = "localhost"
FTPS_PORT = 990
FTPS_USER = "CHU-TEST"
FTPS_PASSWORD = "ftppassword"

# Remote base path on the FTPS server (institution.path in the database).
# Files are uploaded to {REMOTE_PATH}/{PRESCRIPTION_NAME}/ and the ACK is pulled from the same folder.
REMOTE_PATH = "remote/seqoia"

In [ ]:
import ser_client_api
from pathlib import Path
from ser_client_api.hl7v2 import HL7v2Generator, SEQOIA, AURAGEN, PERIGENOMED

# Compiled GIPCAD profiles bundled with ser_client-api; override if using different profiles
_PROFILES_DIR = Path(ser_client_api.__file__).parent / "hl7v2" / "gipcad" / "profiles" / "v000_compiled"
HL7_PROFILE_PATH = _PROFILES_DIR / "oru_r01_lab36"
ACK_PROFILE_PATH = _PROFILES_DIR / "ack_r01_ack"

# Client certificates for FTPS; set to the actual path on your system
CERT_DIR = Path("/path/to/certificates")

# Select the sending institution
SELECTED_INSTITUTION = "seqoia"  # one of: "seqoia", "auragen", "perigenomed"
institution = {"seqoia": SEQOIA, "auragen": AURAGEN, "perigenomed": PERIGENOMED}[SELECTED_INSTITUTION]
generator = HL7v2Generator(HL7_PROFILE_PATH, institution)


## 1. Parse prescription JSON

**Input data**

In [ ]:
# Minimal prescription payload in GLeaves format, trio case (patient + father + mother).

# To read from a JSON file instead, replace the PRESCRIPTION_JSON definition below with:
# PRESCRIPTION_JSON = json.loads(Path("/path/to/prescription.json").read_text())

PRESCRIPTION_JSON = {
    "_id": "67cecdfdb4cffa55e8707002",
    "preindication": {
        "catname": "Maladie rare",
        "catkey": "p1",
        "name": "Angiodèmes bradykiniques héréditaires",
        "key": "p1-sp60",
        "rcp_id": "5e71f8b3efaa9b6f5a729fa6",
        "rcp_nom": "MaRIH-NEUTROPENIES",
    },
    "patients": [
        {
            "patient": {
                "id": {"type": "IPP", "value": "IPP-DEMO-001"},
                "date_naissance": "1980-06-15",
                "sexe": "F",
                "nom": "DUPONT",
                "nom_naissance": "DUPONT",
                "prenom": "MARIE",
                },
            "lien": {"key": "patient", "name": "Patient"},
            "date_prelevement": 1741651200000,
            "is_data_reusable_for_research": True,
            "dateConsent": 1741940668351,
            "id_anon": "66EFA7",
            "resultat_compte_rendu_gleaves": {},
        },
        {
            "patient": {
                "id": {"type": "IPP", "value": "IPP-DEMO-002"},
                "date_naissance": "1955-03-22",
                "sexe": "M",
                "nom": "DUPONT",
                "nom_naissance": "DUPONT",
                "prenom": "JEAN",
            },
            "lien": {"key": "père", "name": "Père"},
            "is_data_reusable_for_research": False,
            "date_prelevement": 1741651200000,
            "id_anon": "66EFA8",
        },
        {
            "patient": {
                "id": {"type": "IPP", "value": "IPP-DEMO-003"},
                "date_naissance": "1958-11-04",
                "sexe": "F",
                "nom": "MARTIN",
                "nom_naissance": "MARTIN",
                "prenom": "CLAIRE",
            },
            "lien": {"key": "mère", "name": "Mère"},
            "is_data_reusable_for_research": False,
            "date_prelevement": 1741651200000,
            "id_anon": "66EFA9",
        },
    ],
    "prescripteur": "4031575",
    "membreRCP": "541351",
    "analysis_info": {
        "analysis_ID": "55014",
        "date_fin_analyse": "08042025",
    },
    "date_creation": 1741606397865,
    "date_cloture": 1745498322775,
    "resultats": [
        {
            "type": "cr_biologique",
            "MembreLMG": "541351",
            "filename": "CR_SeqOIA_Demo.pdf",
        }
    ],
}


**Validate schema**

In [ ]:
from ser_client_api import ParserFactory

parser = ParserFactory(institution).create()
parser.validate(PRESCRIPTION_JSON)
print("Schema validation passed.")

**Parse**

In [ ]:
from ser_client_api.demo import print_composition
from ser_client_api.hl7v2.domain_models import CompositionData

composition: CompositionData = parser.parse(PRESCRIPTION_JSON)
print_composition(composition)

## 2. Prepare files to transfer

In production, genomic data files may come from big data storage infrastructures (downloaded from S3, restored from tape data storage...).

For this demo, small synthetic files are created with a given directory structure.

In [ ]:
from ser_client_api.demo import get_prescription_directory, populate_temporary_presc_dir, print_transfer_directory

PRESCRIPTION_NAME = "DEMO-PRESCRIPTION-001"

tmp_dir, presc_dir = get_prescription_directory(PRESCRIPTION_NAME)
populate_temporary_presc_dir(presc_dir, PRESCRIPTION_NAME, composition)
print_transfer_directory(presc_dir)

## 3. Generate HL7v2 message

Builds an ORU_R01 message from the `CompositionData` and writes it to the transfer directory.  
OBX segments are populated from the files prepared above.

In [ ]:
from ser_client_api.hl7v2 import generate_sidecars

count = generate_sidecars(presc_dir)
print(f"SHA-256 sidecars written: {count}")

In [ ]:
from ser_client_api.demo import print_hl7_file

hl7_file = generator.generate_and_seal(composition, presc_dir, PRESCRIPTION_NAME)
print_hl7_file(hl7_file)

## 4. Transfer files to ser_server-ftps

In [ ]:
import os
from ser_client_ftps import SecureFTPSClient
from ser_client_ftps.exceptions import FTPSConnectionError, FTPSTransferError

client = SecureFTPSClient(
    host=FTPS_HOST,
    port=FTPS_PORT,
    cert_file=str(CERT_DIR / "client-cert.pem"),
    key_file=str(CERT_DIR / "client-key.pem"),
    ca_file=str(CERT_DIR / "ca-cert.pem"),
)

REMOTE_DIR = f"{REMOTE_PATH}/{PRESCRIPTION_NAME}"
files_transferred = []

try:
    with client.connect(FTPS_USER, FTPS_PASSWORD) as host:
        print(f"Connected to {FTPS_HOST}:{FTPS_PORT}")

        if not host.path.exists(REMOTE_DIR):
            host.makedirs(REMOTE_DIR)
            print(f"Created remote directory: {REMOTE_DIR}")

        for local_file in sorted(presc_dir.rglob("*")):
            if not local_file.is_file():
                continue

            relative = local_file.relative_to(presc_dir)
            remote_file = f"{REMOTE_DIR}/{relative}"

            remote_subdir = os.path.dirname(remote_file)
            if remote_subdir and not host.path.exists(remote_subdir):
                host.makedirs(remote_subdir)

            host.upload(str(local_file), remote_file)

            remote_size = host.path.getsize(remote_file)
            local_size = local_file.stat().st_size
            assert remote_size == local_size, f"Size mismatch on {relative}"

            files_transferred.append(remote_file)
            print(f"  ok {relative}  ({remote_size} bytes)")

        print(f"\nDone: {len(files_transferred)} files transferred")

except FTPSConnectionError as e:
    print(f"Connection error (is ser_server-ftps running?): {e}")
except FTPSTransferError as e:
    print(f"Transfer error: {e}")

In [ ]:
# All files are uploaded. Signal to ser_server-ftps that the transfer is complete
# by uploading the .hl7.ok marker - the server will not process the HL7 message before it appears.
try:
    with client.connect(FTPS_USER, FTPS_PASSWORD) as host:
        hl7_ok_local = presc_dir / f"{PRESCRIPTION_NAME}.hl7.ok"
        hl7_ok_local.write_text("")
        hl7_ok_remote = f"{REMOTE_DIR}/{PRESCRIPTION_NAME}.hl7.ok"
        host.upload(str(hl7_ok_local), hl7_ok_remote)
        print(f"Uploaded trigger: {PRESCRIPTION_NAME}.hl7.ok")

except FTPSConnectionError as e:
    print(f"Connection error (is ser_server-ftps running?): {e}")
except FTPSTransferError as e:
    print(f"Transfer error: {e}")

## 5. Verify - list transferred files on the server

In [ ]:
with client.connect(FTPS_USER, FTPS_PASSWORD) as host:
    print(f"Remote /{REMOTE_DIR}/:")
    for entry in sorted(host.listdir(REMOTE_DIR)):
        remote_path = f"{REMOTE_DIR}/{entry}"
        if host.path.isdir(remote_path):
            print(f"  {entry}/")
            for sub in sorted(host.listdir(remote_path)):
                size = host.path.getsize(f"{remote_path}/{sub}")
                print(f"    {sub}  ({size} bytes)")
        else:
            size = host.path.getsize(remote_path)
            print(f"  {entry}  ({size} bytes)")

## 6. Retrieve and parse ACK response

After the transfer, `ser_server-ftps` processes the HL7v2 message and writes an ACK file back to the remote folder.
We pull it via FTPS and parse it with `ser_client_api`.

In [ ]:
ack_filename, ack_content = client.pull_ack(
    username=FTPS_USER,
    password=FTPS_PASSWORD,
    remote_folder_path=REMOTE_DIR,
    institution_name="demo",
)

if ack_filename is None:
    raise RuntimeError("No ACK file found on server - has ser_server-ftps finished processing?")

print(f"Pulled ACK: {ack_filename}")

In [ ]:
from ser_client_api import parse_hl7_message_robust
from ser_client_api.demo import print_ack

ack_msg = parse_hl7_message_robust(ack_content, str(ACK_PROFILE_PATH))
print_ack(ack_msg, ack_filename)

## 7. Cleanup

In [ ]:
import shutil

shutil.rmtree(tmp_dir)
print(f"Removed local temp dir: {tmp_dir}")

### Optional: remove remote files

Run this before re-running the transfer cell to avoid `Overwrite permission denied` errors.

> **Note:** this is only possible against the mock server, not in production.

In [ ]:
with client.connect(FTPS_USER, FTPS_PASSWORD) as host:
    if not host.path.exists(REMOTE_DIR):
        print(f"Not found on server: {REMOTE_DIR}")
    else:
        for entry in host.listdir(REMOTE_DIR):
            remote_path = f"{REMOTE_DIR}/{entry}"
            if host.path.isdir(remote_path):
                for sub in host.listdir(remote_path):
                    host.remove(f"{remote_path}/{sub}")
                host.rmdir(remote_path)
            else:
                host.remove(remote_path)
        host.rmdir(REMOTE_DIR)
        print(f"Removed remote: {REMOTE_DIR}")